# TRM Compiler Pass Ordering — Complete Pipeline

Run this notebook in Google Colab to train TRM and benchmark against real LLVM.

**Runtime:** Select GPU (T4 or higher) or CPU


In [ ]:
# @title Step 1: Run Complete Pipeline
# @markdown This cell runs the entire pipeline:
# @markdown 1. Clones repo (if needed)
# @markdown 2. Installs dependencies (torch, numpy)
# @markdown 3. Trains TRM on synthetic LLVM
# @markdown 4. Benchmarks against real LLVM (if CompilerGym available)

import subprocess
import sys
import os
import warnings
warnings.filterwarnings("ignore")

PROJECT_DIR = "/content/trm-youtubevids"

def setup_environment():
    """Install minimal dependencies for Colab."""
    print("=" * 60)
    print("STEP 1: Installing dependencies")
    print("=" * 60)
    
    # Uninstall potentially conflicting packages
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "numpy", "torch", "-y", "-q"], capture_output=True)
    
    # Install numpy (compatible version)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy>=1.26.0,<2.0", "-q"], capture_output=True)
    
    # Install torch
    subprocess.run([sys.executable, "-m", "pip", "install", "torch", "--index-url", "https://download.pytorch.org/whl/cpu", "-q"], capture_output=True)
    
    # Try to install compiler_gym (continue on error)
    result = subprocess.run([sys.executable, "-m", "pip", "install", "compiler_gym", "-q"], capture_output=True)
    if result.returncode != 0:
        print("Note: compiler_gym installation may have failed, will use synthetic mode")
    
    # Pin numpy
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy>=1.26.0,<2.0", "-q"], capture_output=True)
    
    print("Dependencies installed!")

def clone_repo():
    """Clone or update the repo."""
    print("=" * 60)
    print("STEP 2: Setting up project")
    print("=" * 60)
    
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["COMPILER_GYM_HOME"] = "/content/compiler_gym"
    
    if not os.path.exists(PROJECT_DIR):
        subprocess.run(["git", "clone", "https://github.com/Cree0618/trm-youtubevids.git", PROJECT_DIR], check=True, capture_output=True)
    else:
        subprocess.run(["git", "-C", PROJECT_DIR, "pull"], check=True, capture_output=True)

def verify_compiler_gym():
    """Verify CompilerGym works."""
    print("=" * 60)
    print("STEP 3: Verifying CompilerGym")
    print("=" * 60)
    
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["COMPILER_GYM_HOME"] = "/content/compiler_gym"
    
    try:
        import compiler_gym
    except ImportError as e:
        print(f"compiler_gym not available: {e}")
        print("Using synthetic mode for training")
        return False
    
    try:
        env = compiler_gym.make("llvm-v0", benchmark="cbench-v1/qsort",
            observation_space="Autophase", reward_space="IrInstructionCountOz")
        obs = env.reset()
        ic = obs["IrInstructionCount"]
        print(f"Initial inst: {ic}")
        for i in range(3):
            action = env.action_space.sample()
            obs, reward, done, info = env.step(action)
            inst = obs["IrInstructionCount"]
            print(f"  Step {i}: inst={inst} reward={reward:.2f}")
        env.close()
        print("CompilerGym OK!")
        return True
    except Exception as e:
        print(f"CompilerGym error: {e}")
        print("Using synthetic mode for training")
        return False

def train_and_benchmark(use_synthetic=True):
    """Train TRM and benchmark."""
    print("=" * 60)
    print("STEP 4: Training TRM + Benchmarking")
    print("=" * 60)
    
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["COMPILER_GYM_HOME"] = "/content/compiler_gym"
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    
    # Build args
    sys.argv = [
        "trm_compiler_real_llvm.py",
        "--epochs", "5",
        "--episodes", "5",
        "--benchmarks", "qsort", "adpcm",
        "--max-steps", "20",
        "--batch-size", "64",
        "--num-random", "20",
        "--seed", "42"
    ]
    
    # Add synthetic flag if needed
    if use_synthetic:
        sys.argv.append("--synthetic")
    
    sys.path.insert(0, PROJECT_DIR)
    
    from trm_compiler_real_llvm import main as train_main
    train_main()

def main():
    clone_repo()
    setup_environment()
    
    # Check if CompilerGym works
    cg_ok = verify_compiler_gym()
    
    # Train (use synthetic mode if CompilerGym failed)
    train_and_benchmark(use_synthetic=not cg_ok)
    
    print("=" * 60)
    print("DONE!")
    print("=" * 60)

main()
